In [ ]:
# papermill parameters
COLLECTION  = "OEB"     # "OEB" or "OE"
WORK_ROOT   = "/work"
SAVE        = False     # preview first; set True to write the normalised tables
PREVIEW_SAMPLE = 25
RUN_CONTEXT = "manual"


In [ ]:
# === data.ipynb — loading & normalisation (preview-first, no save by default) ===
#
# What changed in S1 (D-022): this notebook no longer builds a path and no longer defines
# the feature functions. Paths come from utils.run_context.data_paths(COLLECTION); the
# normalisation lives in utils.corpus_prep, where it is tested against the OEB tables it
# originally produced. Setting COLLECTION is now the whole of "run this on another corpus".
#
# Inputs   {collection}_resumen.json, {collection}_texto.json,
#          and any synthetic query sets the collection has ({collection}_single_texto.json …)
# Outputs  {collection}_short_norm.parquet, {collection}_long_norm.parquet,
#          {collection}_{queryset}_norm.parquet per synthetic query set

import sys
from pathlib import Path

for candidate in (Path.cwd(), Path(WORK_ROOT) / "src"):
    if (candidate / "utils").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import pandas as pd
from IPython.display import HTML, display

from utils.corpus_prep import (
    QUERY_FIELDS,
    add_param_token_columns,
    add_processed_columns,
    assert_gold_present,
    axis_labels_from_corpus,
    build_param_tokens,
    clean_df,
    load_records,
    normalize_parameters_field,
)
from utils.run_context import data_paths

paths = data_paths(COLLECTION, work_root=WORK_ROOT)
print(f"collection : {paths.collection}")
print(f"short      : {paths.short_json}")
print(f"long       : {paths.long_json}")
print(f"query sets : {sorted(paths.query_json) or 'none (this collection has no synthetic sets)'}")

# ---------------- the corpus ----------------
short_raw = clean_df(load_records(paths.short_json))
long_raw = clean_df(load_records(paths.long_json))

required = ["item_key", "parent_key", "ud", "concept", "parameters", "text"]
for name, frame in (("short", short_raw), ("long", long_raw)):
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise KeyError(f"{name}: required columns missing: {missing}")

short_proc = add_param_token_columns(add_processed_columns(short_raw, "text"))
long_proc = add_param_token_columns(add_processed_columns(long_raw, "text"))

corpus_keys = set(long_proc["item_key"].astype(str))
assert_gold_present(short_proc, corpus_keys)

# The letter-to-label map lives only in the corpus: the concept schema lists axes by label
# without the letter the records key on. Synthetic queries carry a flat {letter: value} dict,
# so they need this to mint the same param tokens as their gold leaf.
axis_labels = axis_labels_from_corpus(long_proc)

# ---------------- the synthetic query sets ----------------
def prepare_query_set(path: Path) -> pd.DataFrame:
    """Normalise one synthetic query set, keeping its gold and its slice metadata."""
    frame = clean_df(load_records(path, extra_fields=QUERY_FIELDS))

    # Fail loud before anything else: a query whose gold leaf is absent from the corpus is
    # never dropped silently (INTAKE §4 breakpoint 1).
    assert_gold_present(frame, corpus_keys)

    frame = add_processed_columns(frame, "text")

    # add_processed_columns normalises parameters without the axis labels, which a flat query
    # dict does not carry. Redo that part with the labels, so the tokens are comparable.
    normalised = [
        normalize_parameters_field(par, axis_labels.get(str(parent), {}))
        for par, parent in zip(frame["parameters"], frame["parent_key"])
    ]
    frame["parameters_norm"] = [n for n, _ in normalised]
    frame["param_values_multi_norm"] = [m for _, m in normalised]
    return add_param_token_columns(frame)


query_sets = {name: prepare_query_set(path) for name, path in sorted(paths.query_json.items())}

# ---------------- stats ----------------
def quick_stats(df: pd.DataFrame, name: str) -> None:
    lens = df["tokens_word"].map(len)
    numbers = df["numbers"].map(len)
    print(f"{name:16} rows={len(df):6} | tokens avg={lens.mean():6.2f} | %with numbers={100*(numbers>0).mean():6.2f}")

quick_stats(short_proc, "SHORT (resumen)")
quick_stats(long_proc, "LONG (texto)")
for name, frame in query_sets.items():
    quick_stats(frame, name)
    counts = frame["modification_count"].value_counts().sort_index().to_dict()
    print(f"{'':16} gold leaves={frame['gold_item_key'].nunique()} | modification_count={counts}")

# ---------------- preview ----------------
def show_preview(df: pd.DataFrame, title: str, rows: int = 25, max_cols: int = 12) -> None:
    n = min(rows, len(df))
    small = df.iloc[:n, :max_cols].copy()
    for c in small.columns:
        if small[c].dtype == object and c.startswith("numbers"):
            small[c] = small[c].map(lambda x: x if not isinstance(x, list) else (x[:8] + ["…"] if len(x) > 8 else x))
    display(HTML(f"<h3 style='margin:8px 0'>{title}</h3>" + small.to_html(escape=False, index=False, max_rows=n)))

merged = short_proc[["id", "item_key", "parent_key", "ud", "concept", "parameters",
                     "parameters_norm", "param_values_multi_norm",
                     "text_word", "text_word_params", "text_char", "numbers", "param_tokens"]].merge(
    long_proc[["id", "item_key", "parameters_norm", "param_values_multi_norm",
               "text_word", "text_word_params", "text_char", "numbers", "param_tokens"]],
    on="item_key",
    suffixes=("_short", "_long"),
)
show_preview(merged, f"Preview ({min(PREVIEW_SAMPLE, len(merged))} rows) — {COLLECTION} short vs long", PREVIEW_SAMPLE)
for name, frame in query_sets.items():
    show_preview(frame, f"Preview — {COLLECTION} {name}", PREVIEW_SAMPLE)

print("\nPreview generated. To write the tables, set SAVE=True and re-run.")

# ---------------- save ----------------
if SAVE:
    written = []
    short_proc.to_parquet(paths.short_norm, index=False)
    long_proc.to_parquet(paths.long_norm, index=False)
    written += [paths.short_norm, paths.long_norm]
    for name, frame in query_sets.items():
        frame.to_parquet(paths.query_norm[name], index=False)
        written.append(paths.query_norm[name])
    for path in written:
        print(f"saved {path} ({path.stat().st_size:,} bytes)")
